# Unified Evaluation Notebook

This notebook:
1. Loads all 4 saved models (crop recommender, disease detector, yield predictor, pest risk)
2. Runs test predictions on sample Kerala farmer inputs
3. Displays results in a clean formatted output
4. Shows model metrics side by side (if metric files/data are available)
5. Generates a simple **PDF report**


In [ ]:
!pip -q install joblib pandas numpy matplotlib seaborn fpdf2 tensorflow

from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_PATH = '/content/drive/MyDrive/krishi_mitra_models/'
DATA_PATH = '/content/drive/MyDrive/krishi_mitra_data/'

import os
from pathlib import Path
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)


## 1) Load model artifacts

We try local repo paths first, then fallback to Drive paths.


In [ ]:
import json
import joblib
import tensorflow as tf

def pick_existing(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

crop_model_path = pick_existing(
    'ml_models/1_crop_recommender/model/crop_model.pkl',
    Path(MODEL_SAVE_PATH) / 'crop_recommender' / 'crop_model.pkl'
)
crop_le_path = pick_existing(
    'ml_models/1_crop_recommender/model/label_encoder.pkl',
    Path(MODEL_SAVE_PATH) / 'crop_recommender' / 'label_encoder.pkl'
)
yield_model_path = pick_existing(
    'ml_models/3_yield_predictor/model/yield_model.pkl',
    Path(MODEL_SAVE_PATH) / 'yield_predictor' / 'yield_model.pkl'
)
pest_model_path = pick_existing(
    'ml_models/4_pest_risk_forecaster/model/pest_model.pkl',
    Path(MODEL_SAVE_PATH) / 'pest_risk_forecaster' / 'pest_model.pkl'
)
disease_model_path = pick_existing(
    'ml_models/2_disease_detector/model/disease_model.h5',
    Path(MODEL_SAVE_PATH) / 'disease_detector' / 'disease_model.h5'
)
class_names_path = pick_existing(
    'ml_models/2_disease_detector/model/class_names.json',
    Path(MODEL_SAVE_PATH) / 'disease_detector' / 'class_names.json'
)

assert crop_model_path and crop_le_path, 'Crop model artifacts not found. Train Model 1 first.'
assert yield_model_path, 'Yield model artifacts not found. Train Model 3 first.'
assert pest_model_path, 'Pest model artifacts not found. Train Model 4 first.'

crop_model = joblib.load(crop_model_path)
crop_le = joblib.load(crop_le_path)
yield_model = joblib.load(yield_model_path)
pest_model = joblib.load(pest_model_path)

disease_model = None
class_names = None
if disease_model_path and class_names_path:
    disease_model = tf.keras.models.load_model(disease_model_path)
    class_names = json.loads(Path(class_names_path).read_text(encoding='utf-8'))

print('Loaded crop model:', crop_model_path)
print('Loaded yield model:', yield_model_path)
print('Loaded pest model:', pest_model_path)
print('Loaded disease model:', bool(disease_model))


## 2) Sample test input (Kerala)


In [ ]:
test_farmer = {
    'district': 'Palakkad',
    'soil_type': 'Laterite',
    'N': 90, 'P': 42, 'K': 43,
    'pH': 6.5,
    'rainfall': 200,
    'temperature': 28,
    'humidity': 82,
    'season': 'Kharif',
    'water_source': 'Canal',
    'crop': 'Paddy',
    'days_since_sowing': 45,
    'area_hectares': 1.5,
}
test_farmer

## 3) Crop recommendation (top 5)


In [ ]:
import numpy as np
import pandas as pd

X_crop = pd.DataFrame([{
    'district': test_farmer['district'],
    'soil_type': test_farmer['soil_type'],
    'N': test_farmer['N'],
    'P': test_farmer['P'],
    'K': test_farmer['K'],
    'ph': test_farmer['pH'],
    'rainfall': test_farmer['rainfall'],
    'temperature': test_farmer['temperature'],
    'humidity': test_farmer['humidity'],
    'season': test_farmer['season'],
    'water_source': test_farmer['water_source'],
}])

proba = crop_model.predict_proba(X_crop)[0]
idx = np.argsort(proba)[::-1][:5]
top5 = pd.DataFrame({
    'crop': crop_le.inverse_transform(idx),
    'suitability_score': proba[idx].round(3),
})
top5

## 4) Yield prediction


In [ ]:
X_yield = pd.DataFrame([{
    'crop_name': test_farmer['crop'],
    'district': test_farmer['district'],
    'year': 2025,
    'area_hectares': test_farmer['area_hectares'],
    'annual_rainfall_mm': 2800,
    'avg_temperature': test_farmer['temperature'],
    'soil_quality_index': 7,
    'fertilizer_kg_per_hectare': 140,
    'pesticide_kg_per_hectare': 3.0,
    'irrigation_type': 'Canal',
    'season': test_farmer['season'],
    'previous_year_yield': 3.5,
    'avg_3yr_yield': 3.6,
    'drought_index': 0.2,
    'flood_risk_score': 0.6,
}])

yield_pred = float(yield_model.predict(X_yield)[0])
yield_pred

## 5) Pest risk forecast


In [ ]:
X_pest = pd.DataFrame([{
    'district': test_farmer['district'],
    'crop': test_farmer['crop'],
    'month': 7,
    'avg_temperature': test_farmer['temperature'],
    'humidity_percent': test_farmer['humidity'],
    'rainfall_last_7days_mm': 90,
    'wind_speed_kmh': 9.0,
    'previous_pest_incident': 1,
    'crop_growth_stage': 'Vegetative',
    'cloudy': 1,
}])

risk = pest_model.predict(X_pest)[0]
risk_proba = pest_model.predict_proba(X_pest)[0]
risk, risk_proba

## 6) Disease detection (optional)

Requires a leaf image (jpg/png). If the disease model isn’t available yet, skip.


In [ ]:
import numpy as np
from PIL import Image

def disease_predict(image_path: str):
    if disease_model is None:
        return None
    img = Image.open(image_path).convert('RGB').resize((224, 224))
    x = np.expand_dims(np.array(img), 0)
    prob = disease_model.predict(x)[0]
    idx = int(np.argmax(prob))
    return {'disease': class_names[idx], 'confidence': float(prob[idx])}

# Example:
# disease_predict('/content/leaf.jpg')

## 7) Side-by-side model metrics (if available)

If you store metrics during training (recommended), you can load them here.
This template shows a simple table; if no metrics exist, it displays `N/A`.


In [ ]:
import pandas as pd
from pathlib import Path

metrics_paths = {
    'crop_recommender': pick_existing('ml_models/1_crop_recommender/model/metrics.json', Path(MODEL_SAVE_PATH) / 'crop_recommender' / 'metrics.json'),
    'disease_detector': pick_existing('ml_models/2_disease_detector/model/metrics.json', Path(MODEL_SAVE_PATH) / 'disease_detector' / 'metrics.json'),
    'yield_predictor': pick_existing('ml_models/3_yield_predictor/model/metrics.json', Path(MODEL_SAVE_PATH) / 'yield_predictor' / 'metrics.json'),
    'pest_risk': pick_existing('ml_models/4_pest_risk_forecaster/model/metrics.json', Path(MODEL_SAVE_PATH) / 'pest_risk_forecaster' / 'metrics.json'),
}

rows = []
for name, p in metrics_paths.items():
    if p and Path(p).exists():
        rows.append({'model': name, **json.loads(Path(p).read_text(encoding='utf-8'))})
    else:
        rows.append({'model': name, 'metrics': 'N/A'})

pd.DataFrame(rows)

## 8) Generate PDF report

Creates `krishi_mitra_model_report.pdf` in your Drive.


In [ ]:
from fpdf import FPDF
from datetime import datetime

pdf = FPDF()
pdf.add_page()
pdf.set_font('Helvetica', size=14)
pdf.cell(0, 10, 'Krishi Mitra — Model Performance Report', ln=True)
pdf.set_font('Helvetica', size=10)
pdf.cell(0, 8, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}", ln=True)

pdf.ln(4)
pdf.set_font('Helvetica', size=12)
pdf.cell(0, 8, 'Sample input:', ln=True)
pdf.set_font('Helvetica', size=9)
for k, v in test_farmer.items():
    pdf.cell(0, 6, f"- {k}: {v}", ln=True)

pdf.ln(3)
pdf.set_font('Helvetica', size=12)
pdf.cell(0, 8, 'Crop recommendation (top 5):', ln=True)
pdf.set_font('Helvetica', size=9)
for _, r in top5.iterrows():
    pdf.cell(0, 6, f"- {r['crop']}: score {r['suitability_score']}", ln=True)

pdf.ln(2)
pdf.set_font('Helvetica', size=12)
pdf.cell(0, 8, 'Yield prediction:', ln=True)
pdf.set_font('Helvetica', size=9)
pdf.cell(0, 6, f"- Predicted tons/hectare: {yield_pred:.3f}", ln=True)

pdf.ln(2)
pdf.set_font('Helvetica', size=12)
pdf.cell(0, 8, 'Pest risk:', ln=True)
pdf.set_font('Helvetica', size=9)
pdf.cell(0, 6, f"- Risk level: {risk}", ln=True)

pdf.ln(2)
pdf.set_font('Helvetica', size=12)
pdf.cell(0, 8, 'Stored metrics (if available):', ln=True)
pdf.set_font('Helvetica', size=9)
for row in rows:
    pdf.multi_cell(0, 5, f"- {row['model']}: {row.get('metrics', row)}")

out_pdf = Path(MODEL_SAVE_PATH) / 'krishi_mitra_model_report.pdf'
pdf.output(str(out_pdf))
print('Wrote:', out_pdf)
